<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_8_model_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_8_model_lstm

LSTM (Long Short-Term Memory)

## Introducción y Resumen

El objetivo de esta notebook ....



## 0. Configuración del Entorno


### 0.1. Instalación de librerías


In [1]:
# añadimos utilidades de torch
!pip -q install torchinfo einops --upgrade


### 0.2. Importación de librerías


In [2]:
import sys, platform, os, random
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

# Utilidades
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [3]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1


In [4]:
# Chequeo de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Usando CPU")


CUDA disponible: True
GPU: Tesla T4


In [5]:
# Seeds para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### 0.3. Acceso a Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [7]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [8]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [9]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [10]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [11]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [12]:
print(f'Listado de features para 30min: {features_to_30}')
print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 30min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [13]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [14]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas 30 minutos

In [15]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [16]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [17]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [18]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [19]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [20]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [21]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [22]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [23]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [24]:
lstm_metrics, metrics = load_or_create_metrics("4_8_lstm_metrics")

Las métricas existen y son almacenadas en lstm_metrics


In [25]:
metrics

True

In [26]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908
LSTM_60_subsampleado_30%,0.004084,0.002402,0.218303,116.811436,0.697755
LSTM_60_subsampleado_50%,0.004080,0.002302,0.228758,115.545740,0.710292
LSTM_60_100%,0.004300,0.002514,0.142216,123.553541,0.667525
LSTM_90_subsampleado_30%,0.004942,0.002759,0.250770,110.116369,0.734964
LSTM_90_subsampleado_50%,0.005318,0.002876,0.231798,112.849430,0.724631
LSTM_90_100%,0.005338,0.002932,0.177724,125.569550,0.693459


### 3.2. Función para guardar métricas

In [27]:
def save_metrics (metrics,  metrics_name: str):   #("4_2_xgboost_metrics")
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [28]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [29]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

In [30]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908
LSTM_60_subsampleado_30%,0.004084,0.002402,0.218303,116.811436,0.697755
LSTM_60_subsampleado_50%,0.004080,0.002302,0.228758,115.545740,0.710292
LSTM_60_100%,0.004300,0.002514,0.142216,123.553541,0.667525
LSTM_90_subsampleado_30%,0.004942,0.002759,0.250770,110.116369,0.734964
LSTM_90_subsampleado_50%,0.005318,0.002876,0.231798,112.849430,0.724631
LSTM_90_100%,0.005338,0.002932,0.177724,125.569550,0.693459


## 4. Definición de modelo


### 4.1. Función de entrenamiento para modelo

In [31]:
from typing import Optional, Dict, Any, Tuple
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def train_model_rnn(
    best_params: Optional[Dict[str, Any]],
    X_train: np.ndarray, y_train: np.ndarray,
    X_valid: np.ndarray, y_valid: np.ndarray,
    *,
    use_internal_early_stopping: bool = True,
    # Arquitectura (RNN)
    rnn_type: str = "lstm",               # "lstm" o "gru"
    hidden_size: int = 128,
    num_layers: int = 2,
    dropout: float = 0.1,
    bidirectional: bool = False,
    # Optimización
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 512,
    max_epochs: int = 50,
    patience: int = 8,
    # Estabilizadores
    grad_clip: Optional[float] = 1.0,     # None para desactivar clipping
    use_scheduler: bool = True,           # ReduceLROnPlateau
    scheduler_factor: float = 0.5,
    scheduler_patience: int = 3,
    scheduler_min_lr: float = 1e-5,
    # Varios
    num_workers: int = 0,
    device: Optional[str] = None,
    verbose: bool = False,
):
    """
    Entrena un RNN (LSTM o GRU) para regresión y devuelve (modelo, preds_valid).
    - X_* debe ser 3D: (n_samples, seq_len, n_features)
    - y_* puede ser 1D o 2D con última dimensión = 1
    - Incluye gradient clipping y ReduceLROnPlateau opcionales.
    - Interfaz y lógica de entrenamiento alineadas con train_model_tcn.
    """

    # -------------------------
    # Defaults + overrides desde best_params
    # -------------------------
    params = dict(best_params or {})
    rnn_type      = str(params.get("rnn_type", rnn_type)).lower()
    hidden_size   = int(params.get("hidden_size", hidden_size))
    num_layers    = int(params.get("num_layers", num_layers))
    dropout       = float(params.get("dropout", dropout))
    bidirectional = bool(params.get("bidirectional", bidirectional))

    lr            = float(params.get("lr", lr))
    weight_decay  = float(params.get("weight_decay", weight_decay))
    batch_size    = int(params.get("batch_size", batch_size))
    max_epochs    = int(params.get("max_epochs", max_epochs))
    patience      = int(params.get("patience", patience))

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    # -------------------------
    # Validaciones de forma
    # -------------------------
    if X_train.ndim != 3 or X_valid.ndim != 3:
        raise ValueError("X_train y X_valid deben ser 3D: (n_samples, seq_len, n_features).")

    y_train = y_train.reshape(-1)
    y_valid = y_valid.reshape(-1)

    n_features = X_train.shape[-1]

    # -------------------------
    # Tensores y loaders
    # -------------------------
    Xtr = torch.tensor(X_train, dtype=torch.float32)
    Ytr = torch.tensor(y_train, dtype=torch.float32)
    Xva = torch.tensor(X_valid, dtype=torch.float32)
    Yva = torch.tensor(y_valid, dtype=torch.float32)

    train_ds = TensorDataset(Xtr, Ytr)
    valid_ds = TensorDataset(Xva, Yva)

    # Nota: dejamos shuffle=False para ser 100% drop-in con tu TCN.
    # Si querés mejorar generalización, podés poner shuffle=True en train_loader.
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    # -------------------------
    # Modelo RNN (LSTM/GRU)
    # -------------------------
    class RNNRegressor(nn.Module):
        def __init__(self, input_size, hidden_size, num_layers, dropout, bidirectional, rnn_type="lstm"):
            super().__init__()
            self.bidirectional = bidirectional
            self.rnn_type = rnn_type.lower()
            effective_dropout = dropout if num_layers > 1 else 0.0
            rnn_cls = nn.LSTM if self.rnn_type == "lstm" else nn.GRU
            self.rnn = rnn_cls(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                dropout=effective_dropout,
                bidirectional=bidirectional
            )
            out_dim = hidden_size * (2 if bidirectional else 1)
            self.head = nn.Linear(out_dim, 1)

        def forward(self, x):  # x: (B, T, F)
            out, _ = self.rnn(x)            # out: (B, T, H*(1|2))
            last = out[:, -1, :]            # último paso temporal
            y = self.head(last)             # (B, 1)
            return y.squeeze(-1)            # (B,)

    model = RNNRegressor(
        input_size=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        bidirectional=bidirectional,
        rnn_type=rnn_type
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=scheduler_factor,
            patience=scheduler_patience, min_lr=scheduler_min_lr
        )

    # -------------------------
    # Entrenamiento + Early Stop
    # -------------------------
    best_state = None
    best_val = float("inf")
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        # ---- train ----
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

            optimizer.step()
            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_ds)

        # ---- valid ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= len(valid_ds)

        if verbose:
            print(f"Epoch {epoch+1:03d}/{max_epochs} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if scheduler is not None:
            scheduler.step(val_loss)

        # Early stopping
        if use_internal_early_stopping:
            if val_loss < best_val - 1e-10:
                best_val = val_loss
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    if verbose:
                        print(f"Early stopping en epoch {epoch+1} (mejor val_loss={best_val:.6f}).")
                    break
        else:
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restaurar mejor estado
    if best_state is not None:
        model.load_state_dict(best_state)

    # -------------------------
    # Predicciones valid
    # -------------------------
    model.eval()
    preds_list = []
    with torch.no_grad():
        for xb, _ in valid_loader:
            xb = xb.to(device)
            preds = model(xb).detach().cpu().numpy()
            preds_list.append(preds)
    preds_valid = np.concatenate(preds_list, axis=0)

    return model, preds_valid


### 4.2. Parámetros por defecto para modelo


In [32]:
import math

def rnn_params_for_setup(
    horizon_minutes: int,         # 15, 20, 30, 60, 90...
    rate_sampled: float,          # 0.3, 0.5, 1.0
    *,
    rnn_type: str = "lstm",       # "lstm" o "gru"
    base_hidden: int = 128,
    base_layers: int = 2,
    causal_bidirectional: bool = False,  # mantener False para no "ver el futuro"
):
    """
    Devuelve hiperparámetros recomendados para LSTM/GRU según horizonte y tamaño de subset.
    No hay ajuste por 'receptive field' como en TCN, pero sí por:
      - capacidad (hidden_size, num_layers)
      - regularización (dropout, weight_decay)
      - estabilidad (lr, batch, epochs, patience)
    """

    # ======================
    # 1) Capacidad base
    # ======================
    hidden_size = base_hidden
    num_layers  = base_layers
    dropout     = 0.10
    weight_decay = 1e-5
    lr          = 1e-3
    batch_size  = 512
    max_epochs  = 60
    patience    = 10
    use_scheduler = True
    scheduler_factor = 0.5
    scheduler_patience = 3
    scheduler_min_lr = 1e-5
    grad_clip   = 1.0
    verbose     = False

    # ======================
    # 2) Ajustes por subset
    #    (más datos -> batch más chico, más epochs; menos datos -> batch grande y lr más alto)
    # ======================
    if rate_sampled <= 0.3 + 1e-9:
        lr = 1e-3
        batch_size = 512
        dropout = 0.10
        weight_decay = 1e-5
        max_epochs = 60
        patience = 10
    elif rate_sampled <= 0.5 + 1e-9:
        lr = 7.5e-4
        batch_size = 384
        dropout = 0.15
        weight_decay = 1e-5
        max_epochs = 70
        patience = 12
    else:  # 100%
        lr = 3e-4
        batch_size = 256
        dropout = 0.20
        weight_decay = 1e-4
        max_epochs = 90
        patience = 15

    # ======================
    # 3) Microajuste por horizonte
    #    (horizontes más largos suelen tener menos ruido instantáneo -> un poco menos de dropout;
    #     horizontes cortos pueden requerir más capacidad/regularización)
    # ======================
    if horizon_minutes >= 90:
        dropout = max(0.05, dropout - 0.05)
        hidden_size = max(96, base_hidden)      # capacidad suficiente
        num_layers  = max(2, base_layers)
    elif horizon_minutes <= 20:
        # tareas más reactivas: podés subir un toque la capacidad o regularización
        hidden_size = base_hidden
        num_layers  = base_layers
        dropout = min(0.25, dropout + 0.05)

    # ======================
    # 4) Elección GRU vs LSTM (pequeños nudges)
    # ======================
    rnn_type = rnn_type.lower()
    if rnn_type == "gru":
        # GRU suele converger un poco más rápido; permitimos lr apenas mayor
        lr *= 1.0
    else:
        rnn_type = "lstm"

    # ======================
    # 5) Empaquetado final
    # ======================
    return {
        "rnn_type": rnn_type,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "bidirectional": bool(causal_bidirectional),  # mantener False para causalidad
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "max_epochs": max_epochs,
        "patience": patience,
        "use_scheduler": use_scheduler,
        "scheduler_factor": scheduler_factor,
        "scheduler_patience": scheduler_patience,
        "scheduler_min_lr": scheduler_min_lr,
        "grad_clip": grad_clip,
        "verbose": verbose
    }

In [33]:
lstm_params_30_30 = rnn_params_for_setup(horizon_minutes=30, rate_sampled=0.3)
lstm_params_30_50 = rnn_params_for_setup(horizon_minutes=30, rate_sampled=0.5)
lstm_params_30_100 = rnn_params_for_setup(horizon_minutes=30, rate_sampled=1.0)

lstm_params_60_30 = rnn_params_for_setup(horizon_minutes=60, rate_sampled=0.3)
lstm_params_60_50 = rnn_params_for_setup(horizon_minutes=60, rate_sampled=0.5)
lstm_params_60_100 = rnn_params_for_setup(horizon_minutes=60, rate_sampled=1.0)

lstm_params_90_30 = rnn_params_for_setup(horizon_minutes=90, rate_sampled=0.3)
lstm_params_90_50 = rnn_params_for_setup(horizon_minutes=90, rate_sampled=0.5)
lstm_params_90_100 = rnn_params_for_setup(horizon_minutes=90, rate_sampled=1.0)

### 4.3. Función conjunta

In [34]:
# Asumimos que ya definiste:
# - subsample(X, y, n)
# - train_model_tcn(best_params, X_train, y_train, X_valid, y_valid, ...)
# - evaluate_model(model, X, y_true, y_pred=None, eps=1e-8)  (opcional)

def _internal_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Fallback si no pasás evaluate_fn. Calcula RMSE, MAE, R2, SMAPE y DirAcc."""
    from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    # SMAPE clásico (en %) con epsilon para evitar div/0
    eps = 1e-12
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(min=eps)
    smape = float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)
    diracc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "SMAPE": smape, "DirAcc": diracc}

def run_rnn_experiment(
    metrics_flag: bool,
    model_key: str,
    X_train_scaled: np.ndarray,
    y_train: np.ndarray,
    X_valid_scaled: np.ndarray,
    y_valid: np.ndarray,
    resample_rate: float,                 # 0.3, 0.5, 1.0
    rnn_params: Dict[str, Any],
    rnn_metrics_df: Optional[pd.DataFrame] = None,
    n_samples_train: Optional[int] = None,
    n_samples_valid: Optional[int] = None,
    *,
    # extras específicos de RNN / PyTorch
    use_internal_early_stopping: bool = True,
    max_epochs: int = 50,
    patience: int = 8,
    verbose: bool = False,
    # soporte opcional para datos 2D → 3D
    enforce_3d_shape: Optional[Tuple[int, int]] = None,
    # hooks opcionales
    evaluate_fn=None,           # si tenés tu evaluate_model(model, X, y, y_pred=...) pasalo acá
    print_metrics_fn=None       # si tenés tu print_metrics(dict, model_key) pasalo acá
) -> dict:
    """
    Ejecuta un experimento LSTM/GRU con (opcional) subsampleo de train/valid.

    Parámetros
    ----------
    metrics_flag : bool
        Si True, no entrena y busca métricas previas en rnn_metrics_df[model_key].
    model_key : str
        Nombre/índice del modelo en la tabla de métricas (ej: 'LSTM_60_sub_50%').
    X_train_scaled, y_train : arrays
        Ventanas y target de entrenamiento (ya escaladas).
        Formato esperado: 3D (n_samples, seq_len, n_features).
        Si vienen 2D (n_samples, seq_len*n_features), usar enforce_3d_shape=(seq_len, n_features).
    X_valid_scaled, y_valid : arrays
        Ventanas y target de validación (ya escaladas), idem formato.
    resample_rate : float
        Proporción a muestrear (0 < r <= 1). 1.0 = sin subsampleo.
    rnn_params : dict
        Hiperparámetros del RNN (rnn_type, hidden_size, num_layers, dropout, lr, etc.).
    rnn_metrics_df : pd.DataFrame | None
        DataFrame de métricas para leer/escribir (index por model_key). Opcional.
    n_samples_train, n_samples_valid : int | None
        Tamaños base para calcular la cantidad a muestrear. Si es None, se infiere de X_*.
    enforce_3d_shape : (window_size, n_features) | None
        Si tus X_* están 2D (aplanadas), se re-forman a 3D con este shape.

    Retorna
    -------
    dict
        {"RMSE","MAE","R2","SMAPE","DirAcc"}.
    """
    # 0) Carga de métricas previas (si corresponde)
    if metrics_flag and (rnn_metrics_df is not None) and (model_key in rnn_metrics_df.index):
        print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
        metrics_dict = rnn_metrics_df.loc[model_key].to_dict()
        if print_metrics_fn is not None:
            print_metrics_fn(metrics_dict, model_key)
        else:
            print(f"[{model_key}] -> {metrics_dict}")
        return metrics_dict

    # 1) Mensaje de entrenamiento
    tag_rate = f"{int(resample_rate*100)}%" if resample_rate < 1.0 else "100%"
    print(f"Entrenando modelo {model_key} (resample={tag_rate})...")

    # 2) Re-shape opcional 2D → 3D
    def _ensure_3d(X: np.ndarray, name: str) -> np.ndarray:
        if X.ndim == 3:
            return X
        if X.ndim == 2 and enforce_3d_shape is not None:
            T, F = enforce_3d_shape
            if X.shape[1] != T * F:
                raise ValueError(f"{name} tiene shape {X.shape} pero enforce_3d_shape={enforce_3d_shape} "
                                 f"no coincide con columnas esperadas={T*F}.")
            return X.reshape(X.shape[0], T, F)
        raise ValueError(f"{name} debe ser 3D (n, T, F) o pasar enforce_3d_shape=(T, F) si viene aplanado.")

    X_train_3d = _ensure_3d(X_train_scaled, "X_train_scaled")
    X_valid_3d = _ensure_3d(X_valid_scaled, "X_valid_scaled")

    # 3) Tamaños base
    if n_samples_train is None:
        n_samples_train = X_train_3d.shape[0]
    if n_samples_valid is None:
        n_samples_valid = X_valid_3d.shape[0]

    # 4) Subsampleo (si aplica)
    if resample_rate < 1.0:
        n_train_sub = max(1, int(n_samples_train * resample_rate))
        n_valid_sub = max(1, int(n_samples_valid * resample_rate))
        X_train_sub, y_train_sub = subsample(X_train_3d, y_train, n_train_sub)
        X_valid_sub, y_valid_sub = subsample(X_valid_3d, y_valid, n_valid_sub)
    else:
        X_train_sub, y_train_sub = X_train_3d, y_train
        X_valid_sub, y_valid_sub = X_valid_3d, y_valid

    # 5) Entrenamiento y predicción en valid (LSTM/GRU)
    model, y_pred_valid = train_model_rnn(
        best_params=rnn_params,
        X_train=X_train_sub, y_train=y_train_sub,
        X_valid=X_valid_sub, y_valid=y_valid_sub,
        use_internal_early_stopping=use_internal_early_stopping,
        max_epochs=max_epochs,
        patience=patience,
        verbose=verbose
    )

    # 6) Evaluación (usa tu evaluate_model si lo pasaste; si no, usa fallback interno)
    if evaluate_fn is not None:
        metrics_dict = evaluate_fn(model, X_valid_sub, y_valid_sub, y_pred=y_pred_valid)
    else:
        metrics_dict = _internal_metrics(np.ravel(y_valid_sub), np.ravel(y_pred_valid))

    # 7) Mostrar
    if print_metrics_fn is not None:
        print_metrics_fn(metrics_dict, model_key)
    else:
        print(f"[{model_key}] -> {metrics_dict}")

    # 8) Persistir métricas en el DataFrame si se pasa
    if rnn_metrics_df is not None:
        rnn_metrics_df.loc[model_key] = metrics_dict

    return metrics_dict

### 4.4. Función para subsamplear

In [35]:
def subsample(X, y, n):
    n = min(n, X.shape[0])
    idx = np.random.choice(X.shape[0], size=n, replace=False)
    return X[idx], y[idx]

In [36]:
#xgb_device_test = xgb.XGBRegressor(tree_method="gpu_hist", predictor="gpu_predictor")
#try:
#    xgb_device_test.fit([[0,0],[1,1]], [0,1])
#    print("✅ XGBoost GPU works correctly")
#except Exception as e:
#    print("❌ GPU not available for XGBoost:", e)

## 5. Entrenamiento

### 5.1. Entrenamiento 30min

#### 5.1.1. Con 30% de dataset

In [37]:
features_base = ['open','high','close','low','volume']

In [40]:
lstm_30_30 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_30_subsampleado_30%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.3,                       # 30% del dataset
    rnn_params=lstm_params_30_30,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_30_30["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_30_30["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.

Entrenando modelo LSTM_30_subsampleado_30% (resample=30%)...
Métricas de LSTM_30_subsampleado_30%:

	 RMSE:	 0.003009
	  MAE:	 0.001754
	   R2:	 0.175675
	SMAPE:	 126.755719
	DirAcc:	 0.645148


In [41]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148


#### 5.1.2. Con 50% de dataset

In [42]:
lstm_30_50 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_30_subsampleado_50%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.5,                       # 30% del dataset
    rnn_params=lstm_params_30_50,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_30_30["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_30_30["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)

Entrenando modelo LSTM_30_subsampleado_50% (resample=50%)...
Métricas de LSTM_30_subsampleado_50%:

	 RMSE:	 0.003029
	  MAE:	 0.001784
	   R2:	 0.152491
	SMAPE:	 126.852899
	DirAcc:	 0.642159


In [38]:
lstm_params_30_50.update({
    "dropout": 0.20,          # 👈 dentro de 0.15–0.25
    "weight_decay": 1e-4,     # 👈 mejora sugerida
})

In [40]:
lstm_30_50 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_30_subsampleado_50%_dropout_wd",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.5,                       # 30% del dataset
    rnn_params=lstm_params_30_50,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_30_30["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_30_30["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)

Entrenando modelo LSTM_30_subsampleado_50%_dropout_wd (resample=50%)...
Métricas de LSTM_30_subsampleado_50%_dropout_wd:

	 RMSE:	 0.003040
	  MAE:	 0.001785
	   R2:	 0.140381
	SMAPE:	 129.918634
	DirAcc:	 0.638262


In [41]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908
LSTM_60_subsampleado_30%,0.004084,0.002402,0.218303,116.811436,0.697755
LSTM_60_subsampleado_50%,0.004080,0.002302,0.228758,115.545740,0.710292
LSTM_60_100%,0.004300,0.002514,0.142216,123.553541,0.667525
LSTM_90_subsampleado_30%,0.004942,0.002759,0.250770,110.116369,0.734964
LSTM_90_subsampleado_50%,0.005318,0.002876,0.231798,112.849430,0.724631
LSTM_90_100%,0.005338,0.002932,0.177724,125.569550,0.693459
LSTM_30_subsampleado_50%_dropout_wd,0.003040,0.001785,0.140381,129.918634,0.638262


#### 5.1.3. Con ventanas completas

In [43]:
lstm_30_100 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_30_100%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=1,                       # 30% del dataset
    rnn_params=lstm_params_30_100,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    #max_epochs=tcn_params_30_30["max_epochs"],                           # podés subirlo un poco
    #patience=tcn_params_30_30["patience"],                             # paciencia del early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_30)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)

Entrenando modelo LSTM_30_100% (resample=100%)...
Métricas de LSTM_30_100%:

	 RMSE:	 0.003129
	  MAE:	 0.001905
	   R2:	 0.085712
	SMAPE:	 127.601722
	DirAcc:	 0.623908


In [44]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908


### 4.2. Entrenamiento 60min

#### 4.2.1. Con 30% de dataset

In [45]:
lstm_60_30 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_60_subsampleado_30%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.3,                       # 30% del dataset
    rnn_params=lstm_params_60_30,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_60)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.

Entrenando modelo LSTM_60_subsampleado_30% (resample=30%)...
Métricas de LSTM_60_subsampleado_30%:

	 RMSE:	 0.004084
	  MAE:	 0.002402
	   R2:	 0.218303
	SMAPE:	 116.811436
	DirAcc:	 0.697755


#### 4.2.2. Con 50% de dataset

In [46]:
lstm_60_50 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_60_subsampleado_50%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.5,                       # 30% del dataset
    rnn_params=lstm_params_60_50,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_60)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)

Entrenando modelo LSTM_60_subsampleado_50% (resample=50%)...
Métricas de LSTM_60_subsampleado_50%:

	 RMSE:	 0.004080
	  MAE:	 0.002302
	   R2:	 0.228758
	SMAPE:	 115.545740
	DirAcc:	 0.710292


In [42]:
lstm_params_60_50.update({
    "dropout": 0.20,          # 👈 dentro de 0.15–0.25
    "weight_decay": 1e-4,     # 👈 mejora sugerida
})

In [43]:
lstm_60_50 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_60_subsampleado_50%_dropout_wd",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.5,                       # 30% del dataset
    rnn_params=lstm_params_60_50,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_60)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)

Entrenando modelo LSTM_60_subsampleado_50%_dropout_wd (resample=50%)...
Métricas de LSTM_60_subsampleado_50%_dropout_wd:

	 RMSE:	 0.004185
	  MAE:	 0.002391
	   R2:	 0.194215
	SMAPE:	 121.350654
	DirAcc:	 0.688495


In [44]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908
LSTM_60_subsampleado_30%,0.004084,0.002402,0.218303,116.811436,0.697755
LSTM_60_subsampleado_50%,0.004080,0.002302,0.228758,115.545740,0.710292
LSTM_60_100%,0.004300,0.002514,0.142216,123.553541,0.667525
LSTM_90_subsampleado_30%,0.004942,0.002759,0.250770,110.116369,0.734964
LSTM_90_subsampleado_50%,0.005318,0.002876,0.231798,112.849430,0.724631
LSTM_90_100%,0.005338,0.002932,0.177724,125.569550,0.693459
LSTM_30_subsampleado_50%_dropout_wd,0.003040,0.001785,0.140381,129.918634,0.638262


#### 4.2.3. Con ventanas completas

In [47]:
lstm_60_100 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_60_100%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=1,                       # 30% del dataset
    rnn_params=lstm_params_60_100,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_60)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.

Entrenando modelo LSTM_60_100% (resample=100%)...
Métricas de LSTM_60_100%:

	 RMSE:	 0.004300
	  MAE:	 0.002514
	   R2:	 0.142216
	SMAPE:	 123.553541
	DirAcc:	 0.667525


In [48]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908
LSTM_60_subsampleado_30%,0.004084,0.002402,0.218303,116.811436,0.697755
LSTM_60_subsampleado_50%,0.004080,0.002302,0.228758,115.545740,0.710292
LSTM_60_100%,0.004300,0.002514,0.142216,123.553541,0.667525


### 4.3. Entrenamiento 90min

#### 4.3.1. Con 30% de dataset

In [49]:
lstm_90_30 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_90_subsampleado_30%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.3,                       # 30% del dataset
    rnn_params=lstm_params_90_30,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_90)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.
#Time: >3mins

Entrenando modelo LSTM_90_subsampleado_30% (resample=30%)...
Métricas de LSTM_90_subsampleado_30%:

	 RMSE:	 0.004942
	  MAE:	 0.002759
	   R2:	 0.250770
	SMAPE:	 110.116369
	DirAcc:	 0.734964


#### 4.3.2. Con 50% de dataset

In [50]:
lstm_90_50 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_90_subsampleado_50%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.5,                       # 30% del dataset
    rnn_params=lstm_params_90_50,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_90)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.
#Time: >3mins

Entrenando modelo LSTM_90_subsampleado_50% (resample=50%)...
Métricas de LSTM_90_subsampleado_50%:

	 RMSE:	 0.005318
	  MAE:	 0.002876
	   R2:	 0.231798
	SMAPE:	 112.849430
	DirAcc:	 0.724631


In [45]:
lstm_params_90_50.update({
    "dropout": 0.20,          # 👈 dentro de 0.15–0.25
    "weight_decay": 1e-4,     # 👈 mejora sugerida
})

In [46]:
lstm_90_50 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_90_subsampleado_50%_dropout_wd",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.5,                       # 30% del dataset
    rnn_params=lstm_params_90_50,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_90)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.
#Time: >3mins

Entrenando modelo LSTM_90_subsampleado_50%_dropout_wd (resample=50%)...
Métricas de LSTM_90_subsampleado_50%_dropout_wd:

	 RMSE:	 0.005054
	  MAE:	 0.002812
	   R2:	 0.226517
	SMAPE:	 115.056909
	DirAcc:	 0.723909


In [47]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908
LSTM_60_subsampleado_30%,0.004084,0.002402,0.218303,116.811436,0.697755
LSTM_60_subsampleado_50%,0.004080,0.002302,0.228758,115.545740,0.710292
LSTM_60_100%,0.004300,0.002514,0.142216,123.553541,0.667525
LSTM_90_subsampleado_30%,0.004942,0.002759,0.250770,110.116369,0.734964
LSTM_90_subsampleado_50%,0.005318,0.002876,0.231798,112.849430,0.724631
LSTM_90_100%,0.005338,0.002932,0.177724,125.569550,0.693459
LSTM_30_subsampleado_50%_dropout_wd,0.003040,0.001785,0.140381,129.918634,0.638262


#### 4.3.3. Con ventanas completas

In [51]:
lstm_90_100 = run_rnn_experiment(
    metrics_flag=metrics,
    model_key="LSTM_90_100%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=1,                       # 30% del dataset
    rnn_params=lstm_params_90_100,           # hiperparámetros base
    rnn_metrics_df=lstm_metrics,              # DataFrame donde guardar métricas
    use_internal_early_stopping=True,        # activa early stopping
    verbose=False,                           # logs de entrenamiento

    # Si tus ventanas están aplanadas (2D), pasá su forma original:
    # enforce_3d_shape=(window_size, n_features)
    enforce_3d_shape=(90, len(features_to_90)+len(features_base)), # ejemplo si window_size=90

    # Hooks genéricos:
    evaluate_fn=evaluate_model,              # misma función de métricas
    print_metrics_fn=print_metrics           # tu función para mostrar métricas
)
#Time 262 segundos.
#Time: >3mins

Entrenando modelo LSTM_90_100% (resample=100%)...
Métricas de LSTM_90_100%:

	 RMSE:	 0.005338
	  MAE:	 0.002932
	   R2:	 0.177724
	SMAPE:	 125.569550
	DirAcc:	 0.693459


## 5. Recuperación de métricas

In [52]:
lstm_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LSTM_30_subsampleado_30%,0.003009,0.001754,0.175675,126.755719,0.645148
LSTM_30_subsampleado_50%,0.003029,0.001784,0.152491,126.852899,0.642159
LSTM_30_100%,0.003129,0.001905,0.085712,127.601722,0.623908
LSTM_60_subsampleado_30%,0.004084,0.002402,0.218303,116.811436,0.697755
LSTM_60_subsampleado_50%,0.004080,0.002302,0.228758,115.545740,0.710292
LSTM_60_100%,0.004300,0.002514,0.142216,123.553541,0.667525
LSTM_90_subsampleado_30%,0.004942,0.002759,0.250770,110.116369,0.734964
LSTM_90_subsampleado_50%,0.005318,0.002876,0.231798,112.849430,0.724631
LSTM_90_100%,0.005338,0.002932,0.177724,125.569550,0.693459


In [53]:
save_metrics(lstm_metrics, "4_8_lstm_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_8_lstm_metrics.parquet
